In [1]:
import cv2
import threading
from ultralytics import YOLO

def run_tracker_in_thread(filename, model, file_index, frame_queue):
    """
    Runs a video file or webcam stream concurrently with the YOLOv8 model using threading.

    This function captures video frames from a given file or camera source and utilizes the YOLOv8 model for object
    tracking. The function runs in its own thread for concurrent processing.

    Args:
        filename (str): The path to the video file or the identifier for the webcam/external camera source.
        model (obj): The YOLOv8 model object.
        file_index (int): An index to uniquely identify the file being processed, used for display purposes.
        frame_queue (queue.Queue): A queue to store the processed frames for display.

    Note:
        Press 'q' to quit the video display window.
    """
    video = cv2.VideoCapture(filename)  # Read the video file

    while True:
        ret, frame = video.read()  # Read the video frames

        # Exit the loop if no more frames in either video
        if not ret:
            break

        # Track objects in frames if available
        results = model.track(frame, persist=True)
        res_plotted = results[0].plot()

        # Put the processed frame in the queue
        frame_queue.put((file_index, res_plotted))

        key = cv2.waitKey(1)
        if key == ord("q"):
            break

    # Release video sources
    video.release()

# Load the models
model1 = YOLO(r"..\models\yolo11m.pt")
model2 = YOLO(r"..\models\yolo11m.pt")

# Define the video files for the trackers
video_file1 = r"..\videos\vehicle-counting-low.mp4"  # Path to video file, 0 for webcam
# video_file2 = 0  # Path to video file, 0 for webcam, 1 for external camera
video_file2 = r"..\videos\vehicle-counting-low.mp4"  # Path to video file, 0 for webcam, 1 for external camera

# Create a queue to store frames
import queue
frame_queue = queue.Queue()

# Create the tracker threads
tracker_thread1 = threading.Thread(target=run_tracker_in_thread, args=(video_file1, model1, 1, frame_queue), daemon=True)
tracker_thread2 = threading.Thread(target=run_tracker_in_thread, args=(video_file2, model2, 2, frame_queue), daemon=True)

# Start the tracker threads
tracker_thread1.start()
tracker_thread2.start()

# Create a window to display the frames
cv2.namedWindow('Tracking_Stream', cv2.WINDOW_NORMAL)

while True:
    # Get frames from the queue
    frames = {}
    while not frame_queue.empty():
        file_index, frame = frame_queue.get()
        frames[file_index] = frame

    # Concatenate frames side by side if both are available
    if 1 in frames and 2 in frames:
        combined_frame = cv2.hconcat([frames[1], frames[2]])
        cv2.imshow('Tracking_Stream', combined_frame)

    # Break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Wait for the tracker threads to finish
tracker_thread1.join()
tracker_thread2.join()

# Clean up and close windows
cv2.destroyAllWindows()



0: 384x640 1 car, 1 truck, 145.0ms
Speed: 4.0ms preprocess, 145.0ms inference, 85.0ms postprocess per image at shape (1, 3, 384, 640)
0: 384x640 1 car, 1 truck, 222.0ms
Speed: 5.0ms preprocess, 222.0ms inference, 12.1ms postprocess per image at shape (1, 3, 384, 640)


0: 384x640 1 car, 1 truck, 28.0ms
Speed: 3.0ms preprocess, 28.0ms inference, 5.0ms postprocess per image at shape (1, 3, 384, 640)
0: 384x640 1 car, 1 truck, 29.0ms
Speed: 11.0ms preprocess, 29.0ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)


0: 384x640 1 car, 1 truck, 28.0ms
Speed: 1.0ms preprocess, 28.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
0: 384x640 1 car, 1 truck, 28.0ms
Speed: 2.0ms preprocess, 28.0ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)


0: 384x640 1 car, 1 truck, 27.8ms
Speed: 2.7ms preprocess, 27.8ms inference, 5.0ms postprocess per image at shape (1, 3, 384, 640)
0: 384x640 1 car, 1 truck, 28.0ms
Speed: 2.0ms preprocess, 28.0ms in